# BKinD on JHMDB (split 1) — Kaggle

Trains BKinD on JHMDB and scores each finished run into `runs/<name>/eval/`, with `config/JHMDB.yaml`
(the experiment standard: batch 32, mixed precision, 30 epochs). One run takes ~4–5 h.

**Setup (once):**
1. Upload `JHMDB_rgb.zip`, `joint_positions.zip` and `splits.zip` as one **private** Kaggle dataset.
2. In this notebook: **Add Input** → your dataset.
3. **Settings:** Accelerator = **GPU T4 x2** (not P100: recent PyTorch builds may not support it); **Internet = On** (needed for the code, packages and ImageNet weights).
4. If the GitHub repo is private: **Add-ons → Secrets**, add `GITHUB_TOKEN` (a GitHub token with read access to the repo).

All the work is done by `scripts/kaggle_bkind.py` in the repo, which is cloned fresh each session: change code there and push, not here. This notebook only chooses the runs.

**Runs** (`RUNS`): each has a unique `name` (its output folder; keep it unchanged when resuming) and a `seed`. They train at the same time, one per GPU. Optional keys:
- `overrides`: settings that differ from the config, e.g. `dict(nkpts=20)`.
- `start_from`: the name of a finished run (attached as input) to continue training from, e.g. with `overrides=dict(epochs=50)`.
- `tune=True`: score on held-out *training* clips only (`eval/metrics_tune.txt`), for choosing settings without looking at the test set.

**Run it in the background:** *Save Version → Save & Run All (Commit)*. It keeps running after you close the browser (up to 12 h), and everything in `/kaggle/working/` is saved as the notebook's output. Training stops itself cleanly before the 12 h limit.

**Resuming a cut-off run:** add the previous version's output as an input (**Add Input → Your Work → this notebook**) and keep the same run names. Each run continues from its last saved epoch.

In [ ]:
# ---- Settings ----
# Tuning round 1 (tune_ep50 needs the version with JHMDB_split1_clean_s1 attached as input).
RUNS = [dict(name='tune_kp20', seed=1, overrides=dict(nkpts=20), tune=True),
        dict(name='tune_ep50', seed=1, overrides=dict(epochs=50), start_from='JHMDB_split1_clean_s1', tune=True)]
# Done: clean baseline, seeds 1 and 2 (original settings).
# RUNS = [dict(name='JHMDB_split1_clean_s1', seed=1),
#         dict(name='JHMDB_split1_clean_s2', seed=2)]
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
MAX_HOURS = 11.5                 # Kaggle ends a session at 12 h

# Get the code (uses the GITHUB_TOKEN secret if the repo is private) and load the helper.
import os, sys, time, subprocess
T0 = time.time()
CODE = '/kaggle/temp/noisykeypoints'
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace('https://', f'https://{UserSecretsClient().get_secret("GITHUB_TOKEN")}@')
except Exception:
    print('no GITHUB_TOKEN secret; cloning as a public repo')
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', url, CODE], check=True)
sys.path.insert(0, f'{CODE}/scripts')
import kaggle_bkind

In [ ]:
runs = kaggle_bkind.run('jhmdb', RUNS, max_hours=MAX_HOURS, t0=T0)